# Read MPI Reproduction Results (no plots)

Prints the contents of every `.npz` written by the reproduction scripts
(`scripts/local/`, `scripts/slurm/`) and manual `mpirun` runs under
`scripts/results/`. No plotting -- just the numbers: experiment, decoder, noise
axes, and a per-grid-point table of logical error rate, error/shot counts, the
Bayes-factor confidence interval (same as `tmcbs.estimate_ler`), timing, and
whether the shot cap was hit.

The second sweep axis is shown in whatever units the run actually used: the
**ebit noise** (`physical x transRatio`, or explicit `--trans-noise`), or the
**ebit-decoherence ratio** (`--ebitt1t2Ratios`).

Set `SEARCH_DIR` in the next cell to narrow the scan to one directory (e.g. the
`manual/` runs, or a single `figN/`).

In [1]:
from pathlib import Path
import re
import sys

import numpy as np


def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "tmcbs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the TMCBS repository or scripts directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import tmcbs

RESULT_ROOT = ROOT / "scripts" / "results"

# Which directory to read. Defaults to every results dir; point it at a subdir
# to narrow the scan, e.g. RESULT_ROOT / "manual" or RESULT_ROOT / "fig3".
SEARCH_DIR = RESULT_ROOT

print(f"Repository root:    {ROOT}")
print(f"Reading .npz under: {SEARCH_DIR}")

Repository root:    /Users/jstack/Documents/tmcbs
Reading .npz under: /Users/jstack/Documents/tmcbs/scripts/results


In [2]:
def _scalar(z, key, default=None):
    # Read a 0-d array entry (experiment / decoder / transRatio) as a Python scalar.
    if key not in z:
        return default
    v = np.asarray(z[key])
    return v.item() if v.ndim == 0 else v


def _is_number(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool)


def _noise_mode(trans, tr):
    # How the runner defined the second sweep axis:
    #   "ratio" : --trans-ratio; transNoiseArr is a [None] placeholder and the real
    #             ebit noise is transRatio * physical (a single column).
    #   "decoh" : --ebitt1t2Ratios; transNoiseArr holds ebit-decoherence ratios
    #             (the ebit infidelity itself is transRatio * physical).
    #   "ebit"  : --trans-noise (or the default); transNoiseArr holds explicit rates.
    if any(t is None for t in trans):
        return "ratio"
    if _is_number(tr):
        return "decoh"
    return "ebit"


def describe_npz(path):
    try:
        rel = path.relative_to(RESULT_ROOT)
    except ValueError:
        rel = path

    print("=" * 92)
    print(f"FILE: {rel}")

    with np.load(path, allow_pickle=True) as z:
        exp = _scalar(z, "experiment")
        dec = _scalar(z, "decoder")
        tr = _scalar(z, "transRatio")
        phys = np.asarray(z["physicalNoiseArr"], float)
        trans = np.asarray(z["transNoiseArr"]).tolist()
        res = np.asarray(z["resultsArr"], float)
        errs = np.asarray(z["totalErrorsArr"], float)
        shots = np.asarray(z["numberOfShotsArr"], float)
        timing = np.asarray(z["timingArr"], float) if "timingArr" in z else None
        shotlim = np.asarray(z["reachedShotLimitArr"]) if "reachedShotLimitArr" in z else None

    exp_name = tmcbs.Experiment(int(exp)).name if exp is not None else "?"
    mode = _noise_mode(trans, tr)

    print(f"  experiment : {exp}  ({exp_name})")
    print(f"  decoder    : {dec}")
    print(f"  transRatio : {tr}")
    if mode == "ratio":
        col_label = "ebit"
        print(f"  noise axis : ebit noise = {float(tr):g} x physical  ('{col_label}' column below)")
    elif mode == "decoh":
        col_label = "decoh"
        infid = f"{float(tr):g} x physical" if _is_number(tr) else "n/a"
        print(f"  noise axis : ebit-decoherence ratio (wait time / T1); ebit infidelity = {infid}")
    else:
        col_label = "ebit"
        print(f"  noise axis : explicit ebit noise ('{col_label}' column below)")
    weight = re.search(r"_w(\d+)", path.stem)
    if weight:
        print(f"  Pauli-string weight (from filename): {weight.group(1)}")
    print(f"  grid shape : {res.shape}   (len(physicalNoiseArr) x len(transNoiseArr))")
    print(f"  physicalNoiseArr : {[f'{p:.3e}' for p in phys]}")
    if mode == "ratio":
        print(f"  transNoiseArr    : [None]  (placeholder; ebit noise set via transRatio)")
    else:
        print(f"  transNoiseArr    : {trans}")
    print()

    cols = (f"  {'phys':>10} {col_label:>11} {'LER':>12} {'errors':>8} {'shots':>13} "
            f"{'CI_low':>12} {'CI_high':>12} {'time_s':>8} {'shotcap':>8}")
    print(cols)
    print("  " + "-" * (len(cols) - 2))

    ni, nj = res.shape
    for i in range(ni):
        for j in range(nj):
            ler, e, s = res[i, j], errs[i, j], shots[i, j]
            if np.isfinite(s) and s > 0 and np.isfinite(e):
                lo, hi = tmcbs.bayes_factor_interval(int(e), int(s))
            else:
                lo, hi = float("nan"), float("nan")
            t = float(timing[i, j]) if timing is not None else float("nan")
            cap = bool(shotlim[i, j]) if shotlim is not None else False
            if mode == "ratio":
                axis_val = phys[i] * float(tr) if _is_number(tr) else float("nan")
            else:
                axis_val = trans[j]
            axis_str = f"{axis_val:.3e}" if _is_number(axis_val) else str(axis_val)
            e_str = str(int(e)) if np.isfinite(e) else "nan"
            s_str = str(int(s)) if np.isfinite(s) else "nan"
            print(f"  {phys[i]:>10.3e} {axis_str:>11} {ler:>12.4e} {e_str:>8} {s_str:>13} "
                  f"{lo:>12.4e} {hi:>12.4e} {t:>8.1f} {str(cap):>8}")
    print()

In [3]:
files = sorted(SEARCH_DIR.rglob("*.npz")) if SEARCH_DIR.exists() else []

if not files:
    print(f"No .npz files found under {SEARCH_DIR}.")
    print("Run a reproduction script (scripts/local/ or scripts/slurm/) or a manual")
    print("mpirun first; results are written under scripts/results/.")
else:
    print(f"Found {len(files)} .npz file(s) under {SEARCH_DIR}:")
    print()
    for f in files:
        describe_npz(f)
    print("=" * 92)
    print(f"Done: {len(files)} file(s).")

Found 1 .npz file(s) under /Users/jstack/Documents/tmcbs/scripts/results:

FILE: manual/cnot_sc_d5.npz
  experiment : 1  (NON_LOCAL_CNOT)
  decoder    : teser
  transRatio : 1.0
  noise axis : ebit noise = 1 x physical  ('ebit' column below)
  grid shape : (1, 1)   (len(physicalNoiseArr) x len(transNoiseArr))
  physicalNoiseArr : ['1.000e-03']
  transNoiseArr    : [None]  (placeholder; ebit noise set via transRatio)

        phys        ebit          LER   errors         shots       CI_low      CI_high   time_s  shotcap
  ------------------------------------------------------------------------------------------------------
   1.000e-03   1.000e-03   4.6475e-04       33         71006   2.2505e-04   8.3331e-04     16.6    False

Done: 1 file(s).
